# One-Class SVM

Learns a boundary around normal windows in feature space; anything outside the boundary is flagged anomalous. RBF kernel, small nu (expected fraction of normal training points allowed to fall outside the boundary).

Trained STRICTLY on `normal` sessions only — motor_off and transition sessions are excluded entirely, so this model's job is simply: does this look like normal running, or not.

 Training:
    - ONLY true normal-running files

#Evaluation:
   - motor_off, motor_being_on, motor_being_off
  - electrical_fault, mechanical_fault, thermal_fault
  - 5-fold file-wise cross-validation
   - final model trained on 100% normal data
 - final evaluation uses 20% normal files and
    one file from every available anomaly type

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
# %pip install pandas numpy scikit-learn joblib openpyxl matplotlib skl2onnx onnx

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import onnx
import pandas as pd

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import OneClassSVM

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType


In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_PATH = r"../../data/Motor_data_model"
SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s", "PWM", "Irms", "Vrms",
    "Current_Variance", "Current_Crest_Factor",
    "Vib_RMS", "Vib_Min", "Vib_Q1", "Vib_Median",
    "Vib_Q3", "Vib_Max", "Vib_Variance", "Vib_Skewness",
]

N_FEATURES = len(FEATURE_COLUMNS)
WINDOW_LEN = 30
WINDOW_STEP = 10
ROW_PERIOD_SEC = 0.015

N_SPLITS = 5
KERNEL = "rbf"
NU = 0.01
GAMMA = "scale"
ANOMALY_SCORE_PERCENTILE = 99.0
FILE_ANOMALY_FRACTION_THRESHOLD = 0.10

OUTPUT_DIR = "pure_normal_one_class_svm_output"
CV_DIR = os.path.join(OUTPUT_DIR, "cross_validation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_model")
FINAL_EVAL_DIR = os.path.join(OUTPUT_DIR, "final_model_evaluation")
MANUAL_TEST_DIR = os.path.join(OUTPUT_DIR, "manual_test_results")

for folder in [CV_DIR, FINAL_DIR, FINAL_EVAL_DIR, MANUAL_TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Features:", N_FEATURES)
print("Window shape:", (WINDOW_LEN, N_FEATURES))
print("Flattened input size:", WINDOW_LEN * N_FEATURES)


In [ ]:
# ==========================================================
# CELL 3 — normal and anomaly file detection
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def normal_state_for(path):
    p, stem = normalized_path(path), Path(path).stem.lower()
    excluded = ["motor_off", "motor_being_on", "motor_being_off", "transition", "off_on_spikes"]
    if any(word in p for word in excluded):
        return None
    return "normal_running" if "/normal/" in p or stem.startswith("normal_") or stem == "normal" else None


def anomaly_label_for(path):
    p = normalized_path(path)
    if "motor_being_on" in p:
        return "motor_being_on"
    if "motor_being_off" in p:
        return "motor_being_off"
    if "/motor_off/" in p or "motor_off" in Path(p).stem.lower():
        return "motor_off"
    if "electrical fault" in p or "abnormal_current" in p:
        return "electrical_fault"
    if "mechanical fault" in p or "abnormal_vibration" in p:
        return "mechanical_fault"
    if "thermal fault" in p or "abnormal_temp" in p:
        return "thermal_fault"
    return None



In [ ]:
# ==========================================================
# CELL 4 — discover and read files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(os.path.join(base_path, "**", "*.xlsx"), recursive=True)
    files += glob.glob(os.path.join(base_path, "**", "*.csv"), recursive=True)
    return sorted(set(files))


def read_one(path):
    path = Path(path)
    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    missing = [c for c in FEATURE_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path.name}: {missing}")

    clean = df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce") / SCALE_DIVISOR
    clean = clean.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(f"{path.name} has {len(clean)} valid rows; need at least {WINDOW_LEN}.")
    return clean


files = discover_files(BASE_PATH)
if not files:
    raise RuntimeError(f"No CSV/XLSX files found under: {BASE_PATH}")

print("Total files discovered:", len(files))


In [ ]:
# ==========================================================
# CELL 5 — load normal and anomaly sessions
# ==========================================================
healthy_sessions, fault_sessions = [], []

for file_path in files:
    normal_state, fault_label = normal_state_for(file_path), anomaly_label_for(file_path)

    if normal_state is None and fault_label is None:
        print("SKIP unrecognized:", os.path.relpath(file_path, BASE_PATH))
        continue

    try:
        df = read_one(file_path)
    except Exception as error:
        print("SKIP read error:", os.path.relpath(file_path, BASE_PATH), "|", error)
        continue

    record = {
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(file_path, BASE_PATH),
        "rows": len(df),
    }

    if normal_state is not None:
        record.update(state=normal_state, binary_label=0)
        healthy_sessions.append(record)
        print(f"N | {normal_state:20s} | {len(df):6d} | {record['file']}")
    else:
        record.update(fault_label=fault_label, binary_label=1)
        fault_sessions.append(record)
        print(f"A | {fault_label:20s} | {len(df):6d} | {record['file']}")

if not healthy_sessions:
    raise RuntimeError("No true normal-running files were loaded.")
if not fault_sessions:
    raise RuntimeError("No anomaly files were loaded.")

print("\nNormal files:", len(healthy_sessions))
print("Fault files:", len(fault_sessions))
print(pd.Series([s["fault_label"] for s in fault_sessions]).value_counts())


In [ ]:
# ==========================================================
# CELL 6 — window helpers
# ==========================================================
def make_windows_from_array(X_raw, scaler):
    X = scaler.transform(X_raw).astype(np.float32)
    starts = np.arange(0, len(X) - WINDOW_LEN + 1, WINDOW_STEP, dtype=np.int64)
    if len(starts) == 0:
        return None, None
    windows = np.stack([X[i:i + WINDOW_LEN] for i in starts]).astype(np.float32)
    return windows, starts


def build_windows_from_sessions(sessions, scaler):
    windows = []
    for session in sessions:
        Xw, _ = make_windows_from_array(session["X"], scaler)
        if Xw is not None:
            windows.append(Xw)
    if not windows:
        return np.empty((0, WINDOW_LEN, N_FEATURES), dtype=np.float32)
    return np.concatenate(windows)


def flatten_windows(X):
    return X.reshape(len(X), -1)


In [ ]:
# ==========================================================
# CELL 7 — One-Class SVM model and scoring
# ==========================================================
def build_one_class_svm():
    return OneClassSVM(kernel=KERNEL, nu=NU, gamma=GAMMA)


def anomaly_score(model, X):
    return (-model.decision_function(flatten_windows(X))).astype(np.float32)


def calibrate_threshold(model, X_normal):
    scores = anomaly_score(model, X_normal)
    return float(np.percentile(scores, ANOMALY_SCORE_PERCENTILE))


In [ ]:
# ==========================================================
# CELL 8 — file-level and window-level predictions
# ==========================================================
def aggregate_predictions(sessions, model, scaler, threshold):
    file_rows, window_rows = [], []

    for session in sessions:
        Xw, starts = make_windows_from_array(session["X"], scaler)
        if Xw is None:
            continue

        scores = anomaly_score(model, Xw)
        flags = (scores > threshold).astype(np.int64)
        state = session.get("state", session.get("fault_label", "unknown"))
        true_label = int(session["binary_label"])
        anomaly_fraction = float(np.mean(flags))
        file_prediction = int(anomaly_fraction >= FILE_ANOMALY_FRACTION_THRESHOLD)

        file_rows.append({
            "file": session["file"],
            "state": state,
            "true_label": true_label,
            "pred_label": file_prediction,
            "windows": len(Xw),
            "anomaly_windows": int(flags.sum()),
            "anomaly_fraction": anomaly_fraction,
            "mean_score": float(scores.mean()),
            "max_score": float(scores.max()),
            "score_threshold": threshold,
        })

        for i, start in enumerate(starts):
            window_rows.append({
                "file": session["file"],
                "state": state,
                "window": i,
                "start_row": int(start),
                "true_label": true_label,
                "pred_label": int(flags[i]),
                "anomaly_score": float(scores[i]),
                "score_threshold": threshold,
            })

    return pd.DataFrame(file_rows), pd.DataFrame(window_rows)



In [ ]:
# ==========================================================
# CELL 9 — build cross-validation table
# ==========================================================
all_sessions = healthy_sessions + fault_sessions

cv_table = pd.DataFrame({
    "index": np.arange(len(all_sessions)),
    "label": [s["binary_label"] for s in all_sessions],
    "group": [s["file"] for s in all_sessions],
})

print(cv_table["label"].value_counts())



In [ ]:
# ==========================================================
# CELL 10 — five-fold file-wise cross-validation
# ==========================================================
splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
cv_metrics, cv_file_results, cv_window_results, cv_test_files = [], [], [], []

for fold, (train_idx, test_idx) in enumerate(
    splitter.split(cv_table, cv_table["label"], groups=cv_table["group"]), start=1
):
    print("\n" + "=" * 70)
    print(f"CV FOLD {fold}/{N_SPLITS}")
    print("=" * 70)

    train_sessions = [all_sessions[i] for i in train_idx]
    test_sessions = [all_sessions[i] for i in test_idx]
    normal_train = [s for s in train_sessions if s["binary_label"] == 0]

    rng = np.random.default_rng(SEED + fold)
    order = rng.permutation(len(normal_train))
    n_cal = max(1, round(0.20 * len(order)))
    cal_ids = set(order[:n_cal].tolist())

    fit_sessions = [s for i, s in enumerate(normal_train) if i not in cal_ids]
    cal_sessions = [s for i, s in enumerate(normal_train) if i in cal_ids]

    scaler = MinMaxScaler().fit(np.concatenate([s["X"] for s in fit_sessions]))
    X_fit = build_windows_from_sessions(fit_sessions, scaler)
    X_cal = build_windows_from_sessions(cal_sessions, scaler)

    model = build_one_class_svm()
    model.fit(flatten_windows(X_fit))
    threshold = calibrate_threshold(model, X_cal)

    file_df, window_df = aggregate_predictions(test_sessions, model, scaler, threshold)
    file_df.insert(0, "fold", fold)
    window_df.insert(0, "fold", fold)

    cv_file_results.append(file_df)
    cv_window_results.append(window_df)

    for s in test_sessions:
        state = s.get("state", s.get("fault_label", "unknown"))
        cv_test_files.append({
            "fold": fold, "file": s["file"], "state": state,
            "true_label": s["binary_label"],
        })
        print(f"label={s['binary_label']} | state={state} | file={s['file']}")

    y_true = window_df["true_label"].to_numpy()
    y_pred = window_df["pred_label"].to_numpy()
    y_score = window_df["anomaly_score"].to_numpy()

    result = {
        "fold": fold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        "score_threshold": threshold,
    }

    cv_metrics.append(result)
    print("\nWINDOW-LEVEL FOLD RESULTS")
    print(pd.Series(result))
    print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"], zero_division=0))



In [ ]:
# ==========================================================
# CELL 11 — combined cross-validation results
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)
cv_test_files_df = pd.DataFrame(cv_test_files)
cv_file_results_df = pd.concat(cv_file_results, ignore_index=True)
cv_window_results_df = pd.concat(cv_window_results, ignore_index=True)

cv_metrics_df.to_csv(os.path.join(CV_DIR, "cv_metrics.csv"), index=False)
cv_test_files_df.to_csv(os.path.join(CV_DIR, "cv_test_files.csv"), index=False)
cv_file_results_df.to_csv(os.path.join(CV_DIR, "cv_file_predictions.csv"), index=False)
cv_window_results_df.to_csv(os.path.join(CV_DIR, "cv_window_predictions.csv"), index=False)

y_true = cv_window_results_df["true_label"].to_numpy()
y_pred = cv_window_results_df["pred_label"].to_numpy()
y_score = cv_window_results_df["anomaly_score"].to_numpy()

overall_metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_true, y_score),
}

print("\nOVERALL CV WINDOW-LEVEL RESULTS")
print(pd.Series(overall_metrics))
print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"], zero_division=0))

pd.DataFrame([overall_metrics]).to_csv(os.path.join(CV_DIR, "overall_metrics.csv"), index=False)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["normal window", "anomaly window"]).plot(values_format="d")
plt.title("Combined CV Window-Level Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(CV_DIR, "confusion_matrix.png"), dpi=180)

plt.show()

# One-Class SVM

Learns a boundary around normal windows in feature space; anything outside the boundary is flagged anomalous. RBF kernel, small nu (expected fraction of normal training points allowed to fall outside the boundary).

Trained STRICTLY on `normal` sessions only — motor_off and transition sessions are excluded entirely, so this model's job is simply: does this look like normal running, or not.

 Training:
    - ONLY true normal-running files

#Evaluation:
   - motor_off, motor_being_on, motor_being_off
  - electrical_fault, mechanical_fault, thermal_fault
  - 5-fold file-wise cross-validation
   - final model trained on 100% normal data
 - final evaluation uses 20% normal files and
    one file from every available anomaly type

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
# %pip install pandas numpy scikit-learn joblib openpyxl matplotlib skl2onnx onnx

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import onnx
import pandas as pd

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import OneClassSVM

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType


In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_PATH = r"../../data/Motor_data_model"
SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s", "PWM", "Irms", "Vrms",
    "Current_Variance", "Current_Crest_Factor",
    "Vib_RMS", "Vib_Min", "Vib_Q1", "Vib_Median",
    "Vib_Q3", "Vib_Max", "Vib_Variance", "Vib_Skewness",
]

N_FEATURES = len(FEATURE_COLUMNS)
WINDOW_LEN = 30
WINDOW_STEP = 10
ROW_PERIOD_SEC = 0.015

N_SPLITS = 5
KERNEL = "rbf"
NU = 0.01
GAMMA = "scale"
ANOMALY_SCORE_PERCENTILE = 99.0
FILE_ANOMALY_FRACTION_THRESHOLD = 0.10

OUTPUT_DIR = "pure_normal_one_class_svm_output"
CV_DIR = os.path.join(OUTPUT_DIR, "cross_validation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_model")
FINAL_EVAL_DIR = os.path.join(OUTPUT_DIR, "final_model_evaluation")
MANUAL_TEST_DIR = os.path.join(OUTPUT_DIR, "manual_test_results")

for folder in [CV_DIR, FINAL_DIR, FINAL_EVAL_DIR, MANUAL_TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Features:", N_FEATURES)
print("Window shape:", (WINDOW_LEN, N_FEATURES))
print("Flattened input size:", WINDOW_LEN * N_FEATURES)


In [ ]:
# ==========================================================
# CELL 3 — normal and anomaly file detection
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def normal_state_for(path):
    p, stem = normalized_path(path), Path(path).stem.lower()
    excluded = ["motor_off", "motor_being_on", "motor_being_off", "transition", "off_on_spikes"]
    if any(word in p for word in excluded):
        return None
    return "normal_running" if "/normal/" in p or stem.startswith("normal_") or stem == "normal" else None


def anomaly_label_for(path):
    p = normalized_path(path)
    if "motor_being_on" in p:
        return "motor_being_on"
    if "motor_being_off" in p:
        return "motor_being_off"
    if "/motor_off/" in p or "motor_off" in Path(p).stem.lower():
        return "motor_off"
    if "electrical fault" in p or "abnormal_current" in p:
        return "electrical_fault"
    if "mechanical fault" in p or "abnormal_vibration" in p:
        return "mechanical_fault"
    if "thermal fault" in p or "abnormal_temp" in p:
        return "thermal_fault"
    return None



In [ ]:
# ==========================================================
# CELL 4 — discover and read files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(os.path.join(base_path, "**", "*.xlsx"), recursive=True)
    files += glob.glob(os.path.join(base_path, "**", "*.csv"), recursive=True)
    return sorted(set(files))


def read_one(path):
    path = Path(path)
    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    missing = [c for c in FEATURE_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path.name}: {missing}")

    clean = df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce") / SCALE_DIVISOR
    clean = clean.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(f"{path.name} has {len(clean)} valid rows; need at least {WINDOW_LEN}.")
    return clean


files = discover_files(BASE_PATH)
if not files:
    raise RuntimeError(f"No CSV/XLSX files found under: {BASE_PATH}")

print("Total files discovered:", len(files))


In [ ]:
# ==========================================================
# CELL 5 — load normal and anomaly sessions
# ==========================================================
healthy_sessions, fault_sessions = [], []

for file_path in files:
    normal_state, fault_label = normal_state_for(file_path), anomaly_label_for(file_path)

    if normal_state is None and fault_label is None:
        print("SKIP unrecognized:", os.path.relpath(file_path, BASE_PATH))
        continue

    try:
        df = read_one(file_path)
    except Exception as error:
        print("SKIP read error:", os.path.relpath(file_path, BASE_PATH), "|", error)
        continue

    record = {
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(file_path, BASE_PATH),
        "rows": len(df),
    }

    if normal_state is not None:
        record.update(state=normal_state, binary_label=0)
        healthy_sessions.append(record)
        print(f"N | {normal_state:20s} | {len(df):6d} | {record['file']}")
    else:
        record.update(fault_label=fault_label, binary_label=1)
        fault_sessions.append(record)
        print(f"A | {fault_label:20s} | {len(df):6d} | {record['file']}")

if not healthy_sessions:
    raise RuntimeError("No true normal-running files were loaded.")
if not fault_sessions:
    raise RuntimeError("No anomaly files were loaded.")

print("\nNormal files:", len(healthy_sessions))
print("Fault files:", len(fault_sessions))
print(pd.Series([s["fault_label"] for s in fault_sessions]).value_counts())


In [ ]:
# ==========================================================
# CELL 6 — window helpers
# ==========================================================
def make_windows_from_array(X_raw, scaler):
    X = scaler.transform(X_raw).astype(np.float32)
    starts = np.arange(0, len(X) - WINDOW_LEN + 1, WINDOW_STEP, dtype=np.int64)
    if len(starts) == 0:
        return None, None
    windows = np.stack([X[i:i + WINDOW_LEN] for i in starts]).astype(np.float32)
    return windows, starts


def build_windows_from_sessions(sessions, scaler):
    windows = []
    for session in sessions:
        Xw, _ = make_windows_from_array(session["X"], scaler)
        if Xw is not None:
            windows.append(Xw)
    if not windows:
        return np.empty((0, WINDOW_LEN, N_FEATURES), dtype=np.float32)
    return np.concatenate(windows)


def flatten_windows(X):
    return X.reshape(len(X), -1)


In [ ]:
# ==========================================================
# CELL 7 — One-Class SVM model and scoring
# ==========================================================
def build_one_class_svm():
    return OneClassSVM(kernel=KERNEL, nu=NU, gamma=GAMMA)


def anomaly_score(model, X):
    return (-model.decision_function(flatten_windows(X))).astype(np.float32)


def calibrate_threshold(model, X_normal):
    scores = anomaly_score(model, X_normal)
    return float(np.percentile(scores, ANOMALY_SCORE_PERCENTILE))


In [ ]:
# ==========================================================
# CELL 8 — file-level and window-level predictions
# ==========================================================
def aggregate_predictions(sessions, model, scaler, threshold):
    file_rows, window_rows = [], []

    for session in sessions:
        Xw, starts = make_windows_from_array(session["X"], scaler)
        if Xw is None:
            continue

        scores = anomaly_score(model, Xw)
        flags = (scores > threshold).astype(np.int64)
        state = session.get("state", session.get("fault_label", "unknown"))
        true_label = int(session["binary_label"])
        anomaly_fraction = float(np.mean(flags))
        file_prediction = int(anomaly_fraction >= FILE_ANOMALY_FRACTION_THRESHOLD)

        file_rows.append({
            "file": session["file"],
            "state": state,
            "true_label": true_label,
            "pred_label": file_prediction,
            "windows": len(Xw),
            "anomaly_windows": int(flags.sum()),
            "anomaly_fraction": anomaly_fraction,
            "mean_score": float(scores.mean()),
            "max_score": float(scores.max()),
            "score_threshold": threshold,
        })

        for i, start in enumerate(starts):
            window_rows.append({
                "file": session["file"],
                "state": state,
                "window": i,
                "start_row": int(start),
                "true_label": true_label,
                "pred_label": int(flags[i]),
                "anomaly_score": float(scores[i]),
                "score_threshold": threshold,
            })

    return pd.DataFrame(file_rows), pd.DataFrame(window_rows)



In [ ]:
# ==========================================================
# CELL 9 — build cross-validation table
# ==========================================================
all_sessions = healthy_sessions + fault_sessions

cv_table = pd.DataFrame({
    "index": np.arange(len(all_sessions)),
    "label": [s["binary_label"] for s in all_sessions],
    "group": [s["file"] for s in all_sessions],
})

print(cv_table["label"].value_counts())



In [ ]:
# ==========================================================
# CELL 10 — five-fold file-wise cross-validation
# ==========================================================
splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
cv_metrics, cv_file_results, cv_window_results, cv_test_files = [], [], [], []

for fold, (train_idx, test_idx) in enumerate(
    splitter.split(cv_table, cv_table["label"], groups=cv_table["group"]), start=1
):
    print("\n" + "=" * 70)
    print(f"CV FOLD {fold}/{N_SPLITS}")
    print("=" * 70)

    train_sessions = [all_sessions[i] for i in train_idx]
    test_sessions = [all_sessions[i] for i in test_idx]
    normal_train = [s for s in train_sessions if s["binary_label"] == 0]

    rng = np.random.default_rng(SEED + fold)
    order = rng.permutation(len(normal_train))
    n_cal = max(1, round(0.20 * len(order)))
    cal_ids = set(order[:n_cal].tolist())

    fit_sessions = [s for i, s in enumerate(normal_train) if i not in cal_ids]
    cal_sessions = [s for i, s in enumerate(normal_train) if i in cal_ids]

    scaler = MinMaxScaler().fit(np.concatenate([s["X"] for s in fit_sessions]))
    X_fit = build_windows_from_sessions(fit_sessions, scaler)
    X_cal = build_windows_from_sessions(cal_sessions, scaler)

    model = build_one_class_svm()
    model.fit(flatten_windows(X_fit))
    threshold = calibrate_threshold(model, X_cal)

    file_df, window_df = aggregate_predictions(test_sessions, model, scaler, threshold)
    file_df.insert(0, "fold", fold)
    window_df.insert(0, "fold", fold)

    cv_file_results.append(file_df)
    cv_window_results.append(window_df)

    for s in test_sessions:
        state = s.get("state", s.get("fault_label", "unknown"))
        cv_test_files.append({
            "fold": fold, "file": s["file"], "state": state,
            "true_label": s["binary_label"],
        })
        print(f"label={s['binary_label']} | state={state} | file={s['file']}")

    y_true = window_df["true_label"].to_numpy()
    y_pred = window_df["pred_label"].to_numpy()
    y_score = window_df["anomaly_score"].to_numpy()

    result = {
        "fold": fold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        "score_threshold": threshold,
    }

    cv_metrics.append(result)
    print("\nWINDOW-LEVEL FOLD RESULTS")
    print(pd.Series(result))
    print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"], zero_division=0))



In [ ]:
# ==========================================================
# CELL 11 — combined cross-validation results
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)
cv_test_files_df = pd.DataFrame(cv_test_files)
cv_file_results_df = pd.concat(cv_file_results, ignore_index=True)
cv_window_results_df = pd.concat(cv_window_results, ignore_index=True)

cv_metrics_df.to_csv(os.path.join(CV_DIR, "cv_metrics.csv"), index=False)
cv_test_files_df.to_csv(os.path.join(CV_DIR, "cv_test_files.csv"), index=False)
cv_file_results_df.to_csv(os.path.join(CV_DIR, "cv_file_predictions.csv"), index=False)
cv_window_results_df.to_csv(os.path.join(CV_DIR, "cv_window_predictions.csv"), index=False)

y_true = cv_window_results_df["true_label"].to_numpy()
y_pred = cv_window_results_df["pred_label"].to_numpy()
y_score = cv_window_results_df["anomaly_score"].to_numpy()

overall_metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_true, y_score),
}

print("\nOVERALL CV WINDOW-LEVEL RESULTS")
print(pd.Series(overall_metrics))
print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"], zero_division=0))

pd.DataFrame([overall_metrics]).to_csv(os.path.join(CV_DIR, "overall_metrics.csv"), index=False)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["normal window", "anomaly window"]).plot(values_format="d")
plt.title("Combined CV Window-Level Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(CV_DIR, "confusion_matrix.png"), dpi=180)

plt.show()

In [ ]:
# ==========================================================
# CELL 12 — train final One-Class SVM on 100% normal data
# ==========================================================
final_scaler = MinMaxScaler().fit(np.concatenate([s["X"] for s in healthy_sessions]))
X_all_healthy = build_windows_from_sessions(healthy_sessions, final_scaler)

final_model = build_one_class_svm()
final_model.fit(flatten_windows(X_all_healthy))

FINAL_SCORE_THRESHOLD = calibrate_threshold(final_model, X_all_healthy)

print("\nFINAL SCORE THRESHOLD:", FINAL_SCORE_THRESHOLD)
print("All pure-normal windows:", X_all_healthy.shape)


In [ ]:
# ==========================================================
# CELL 13 — final model evaluation
# ==========================================================
rng = np.random.default_rng(SEED)

n_normal_eval = max(1, round(0.20 * len(healthy_sessions)))
normal_ids = rng.choice(len(healthy_sessions), size=n_normal_eval, replace=False)
selected_normal = [healthy_sessions[int(i)] for i in normal_ids]

fault_groups = {}
for session in fault_sessions:
    fault_groups.setdefault(session["fault_label"], []).append(session)

selected_faults = [group[int(rng.integers(len(group)))] for group in fault_groups.values()]
evaluation_sessions = selected_normal + selected_faults

print("\nFINAL EVALUATION FILES")
for session in evaluation_sessions:
    state = session.get("state", session.get("fault_label", "unknown"))
    print(f"label={session['binary_label']} | state={state} | file={session['file']}")

eval_file_df, eval_window_df = aggregate_predictions(
    evaluation_sessions, final_model, final_scaler, FINAL_SCORE_THRESHOLD
)

y_true = eval_window_df["true_label"].to_numpy()
y_pred = eval_window_df["pred_label"].to_numpy()
y_score = eval_window_df["anomaly_score"].to_numpy()

final_metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "normal_accuracy": np.mean(y_pred[y_true == 0] == 0),
    "anomaly_accuracy": np.mean(y_pred[y_true == 1] == 1),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_true, y_score),
    "score_threshold": FINAL_SCORE_THRESHOLD,
}

print("\nFINAL WINDOW-LEVEL RESULTS")
print(pd.Series(final_metrics))
print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"], zero_division=0))

eval_file_df.to_csv(os.path.join(FINAL_EVAL_DIR, "file_predictions.csv"), index=False)
eval_window_df.to_csv(os.path.join(FINAL_EVAL_DIR, "window_predictions.csv"), index=False)
pd.DataFrame([final_metrics]).to_csv(os.path.join(FINAL_EVAL_DIR, "metrics.csv"), index=False)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["normal window", "anomaly window"]).plot(values_format="d")
plt.title("Final One-Class SVM Window-Level Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(FINAL_EVAL_DIR, "confusion_matrix.png"), dpi=180)
plt.show()



In [ ]:
# ==========================================================
# CELL 14 — save model, scaler, threshold, and metadata
# ==========================================================
MODEL_PATH = os.path.join(FINAL_DIR, "pure_normal_one_class_svm.pkl")
SCALER_PATH = os.path.join(FINAL_DIR, "pure_normal_scaler.pkl")
THRESHOLD_PATH = os.path.join(FINAL_DIR, "one_class_svm_threshold.npz")
METADATA_PATH = os.path.join(FINAL_DIR, "model_metadata.csv")

joblib.dump(final_model, MODEL_PATH)
joblib.dump(final_scaler, SCALER_PATH)

np.savez(
    THRESHOLD_PATH,
    score_threshold=np.float32(FINAL_SCORE_THRESHOLD),
    file_anomaly_fraction_threshold=np.float32(FILE_ANOMALY_FRACTION_THRESHOLD),
)

pd.DataFrame({
    "feature_id": np.arange(N_FEATURES),
    "feature_name": FEATURE_COLUMNS,
    "scaler_min": final_scaler.data_min_,
    "scaler_max": final_scaler.data_max_,
}).to_csv(METADATA_PATH, index=False)

print("Saved:", MODEL_PATH)
print("Saved:", SCALER_PATH)
print("Saved:", THRESHOLD_PATH)
print("Saved:", METADATA_PATH)



In [ ]:
# ==========================================================
# CELL 15 — export One-Class SVM to ONNX
# ==========================================================
ONNX_PATH = os.path.join(FINAL_DIR, "pure_normal_one_class_svm.onnx")
input_size = WINDOW_LEN * N_FEATURES

onnx_model = convert_sklearn(
    final_model,
    initial_types=[("float_input", FloatTensorType([1, input_size]))],
    target_opset={"": 15, "ai.onnx.ml": 3},
)

onnx.checker.check_model(onnx_model)
onnx.save_model(onnx_model, ONNX_PATH)

print("Saved ONNX model:", ONNX_PATH)
print("ONNX size KB:", round(os.path.getsize(ONNX_PATH) / 1024, 2))


In [ ]:
# ==========================================================
# CELL 15.1 — export complete One-Class SVM as C header
# ==========================================================
HEADER_PATH = os.path.join(FINAL_DIR, "one_class_svm_model.h")

support_vectors = final_model.support_vectors_.astype(np.float32)
dual_coef = final_model.dual_coef_.ravel().astype(np.float32)
intercept = float(final_model.intercept_[0])
gamma = float(final_model._gamma)

with open(HEADER_PATH, "w", encoding="utf-8") as f:
    f.write("#ifndef ONE_CLASS_SVM_MODEL_H\n")
    f.write("#define ONE_CLASS_SVM_MODEL_H\n\n")
    f.write("#include <math.h>\n")
    f.write("#include <stdint.h>\n\n")

    f.write(f"#define OCSVM_N_FEATURES {N_FEATURES}\n")
    f.write(f"#define OCSVM_WINDOW_LEN {WINDOW_LEN}\n")
    f.write(f"#define OCSVM_INPUT_SIZE {WINDOW_LEN * N_FEATURES}\n")
    f.write(f"#define OCSVM_N_SUPPORT {len(support_vectors)}\n")
    f.write(f"#define OCSVM_SCALE_DIVISOR {SCALE_DIVISOR:.1f}f\n")
    f.write(f"#define OCSVM_GAMMA {gamma:.10f}f\n")
    f.write(f"#define OCSVM_INTERCEPT {intercept:.10f}f\n")
    f.write(f"#define OCSVM_SCORE_THRESHOLD {FINAL_SCORE_THRESHOLD:.10f}f\n")
    f.write(
        f"#define OCSVM_FILE_THRESHOLD "
        f"{FILE_ANOMALY_FRACTION_THRESHOLD:.10f}f\n\n"
    )

    f.write("static const float OCSVM_MIN[OCSVM_N_FEATURES] = {\n")
    for value in final_scaler.data_min_:
        f.write(f"    {float(value):.10f}f,\n")
    f.write("};\n\n")

    f.write("static const float OCSVM_MAX[OCSVM_N_FEATURES] = {\n")
    for value in final_scaler.data_max_:
        f.write(f"    {float(value):.10f}f,\n")
    f.write("};\n\n")

    f.write("static const float OCSVM_DUAL_COEF[OCSVM_N_SUPPORT] = {\n")
    for value in dual_coef:
        f.write(f"    {float(value):.10f}f,\n")
    f.write("};\n\n")

    f.write(
        "static const float "
        "OCSVM_SUPPORT_VECTORS[OCSVM_N_SUPPORT][OCSVM_INPUT_SIZE] = {\n"
    )

    for vector in support_vectors:
        values = ", ".join(f"{float(value):.10f}f" for value in vector)
        f.write(f"    {{{values}}},\n")

    f.write("};\n\n")

    f.write(
        "static inline float ocsvm_scale_raw("
        "float raw_value, int feature_id)\n"
    )
    f.write("{\n")
    f.write("    float physical = raw_value / OCSVM_SCALE_DIVISOR;\n")
    f.write("    float minv = OCSVM_MIN[feature_id];\n")
    f.write("    float maxv = OCSVM_MAX[feature_id];\n")
    f.write("    float denominator = maxv - minv;\n")
    f.write("    if (denominator == 0.0f) return 0.0f;\n")
    f.write("    return (physical - minv) / denominator;\n")
    f.write("}\n\n")

    f.write(
        "static inline float ocsvm_decision_function("
        "const float input[OCSVM_INPUT_SIZE])\n"
    )
    f.write("{\n")
    f.write("    float decision = OCSVM_INTERCEPT;\n")
    f.write("    for (int i = 0; i < OCSVM_N_SUPPORT; i++) {\n")
    f.write("        float distance_squared = 0.0f;\n")
    f.write("        for (int j = 0; j < OCSVM_INPUT_SIZE; j++) {\n")
    f.write(
        "            float difference = "
        "input[j] - OCSVM_SUPPORT_VECTORS[i][j];\n"
    )
    f.write("            distance_squared += difference * difference;\n")
    f.write("        }\n")
    f.write(
        "        decision += OCSVM_DUAL_COEF[i] * "
        "expf(-OCSVM_GAMMA * distance_squared);\n"
    )
    f.write("    }\n")
    f.write("    return decision;\n")
    f.write("}\n\n")

    f.write(
        "static inline float ocsvm_anomaly_score("
        "const float input[OCSVM_INPUT_SIZE])\n"
    )
    f.write("{\n")
    f.write("    return -ocsvm_decision_function(input);\n")
    f.write("}\n\n")

    f.write(
        "static inline int ocsvm_is_anomaly("
        "const float input[OCSVM_INPUT_SIZE])\n"
    )
    f.write("{\n")
    f.write(
        "    return ocsvm_anomaly_score(input) "
        "> OCSVM_SCORE_THRESHOLD;\n"
    )
    f.write("}\n\n")

    f.write("#endif\n")

print("Saved header:", HEADER_PATH)
print("Support vectors:", len(support_vectors))
print("Header size KB:", round(os.path.getsize(HEADER_PATH) / 1024, 2))

In [ ]:
# ==========================================================
# CELL 16 — manual test function
# ==========================================================
def manual_one_class_svm_prediction(test_file):
    model = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    threshold_data = np.load(THRESHOLD_PATH)

    threshold = float(threshold_data["score_threshold"])
    file_threshold = float(threshold_data["file_anomaly_fraction_threshold"])

    df = read_one(test_file)
    X_test, starts = make_windows_from_array(df.to_numpy(dtype=np.float32), scaler)
    scores = anomaly_score(model, X_test)
    flags = (scores > threshold).astype(np.int64)

    anomaly_fraction = float(np.mean(flags))
    file_decision = "ANOMALY" if anomaly_fraction >= file_threshold else "HEALTHY"

    result = pd.DataFrame({
        "window": np.arange(len(X_test)),
        "start_row": starts,
        "time_sec": (starts + WINDOW_LEN / 2.0) * ROW_PERIOD_SEC,
        "anomaly_score": scores,
        "score_threshold": threshold,
        "decision": np.where(flags == 1, "anomaly", "normal"),
        "is_anomaly": flags,
    })

    stem = Path(test_file).stem
    output_csv = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}.csv")
    result.to_csv(output_csv, index=False)

    print("\n" + "=" * 60)
    print("MANUAL ONE-CLASS SVM TEST")
    print("=" * 60)
    print("File:", test_file)
    print("Total windows:", len(result))
    print("Normal windows:", int((flags == 0).sum()))
    print("Anomalous windows:", int((flags == 1).sum()))
    print("Anomaly fraction:", round(anomaly_fraction, 6))
    print("File threshold:", file_threshold)
    print("FILE DECISION:", file_decision)
    print("Score threshold:", threshold)
    print("Saved CSV:", output_csv)
    print("\nWindow decisions:")
    print(result["decision"].value_counts())

    plt.figure(figsize=(12, 4))
    plt.plot(result["time_sec"], result["anomaly_score"])
    plt.axhline(threshold, linestyle="--", label="score threshold")
    plt.xlabel("Time [s]")
    plt.ylabel("One-Class SVM anomaly score")
    plt.title(f"One-Class SVM Score — {stem}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_score.png"), dpi=150)
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.step(result["time_sec"], result["is_anomaly"], where="mid")
    plt.yticks([0, 1], ["normal", "anomaly"])
    plt.ylim(-0.1, 1.1)
    plt.xlabel("Time [s]")
    plt.ylabel("Decision")
    plt.title(f"One-Class SVM Decision Timeline — {stem}")
    plt.tight_layout()
    plt.savefig(os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_timeline.png"), dpi=150)
    plt.show()

    return result


In [ ]:
# ==========================================================
# CELL 17 — select manual test file
# ==========================================================
if __name__ == "__main__":
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    TEST_FILE = filedialog.askopenfilename(
        title="Select motor CSV or Excel file",
        filetypes=[
            ("Excel files", "*.xlsx"),
            ("CSV files", "*.csv"),
            ("All files", "*.*"),
        ],
    )

    root.destroy()

    if TEST_FILE:
        manual_result = manual_one_class_svm_prediction(TEST_FILE)
        print(manual_result.head(30))
    else:
        print("No file selected.")

In [ ]:
# ==========================================================
# CELL 12 — train final One-Class SVM on 100% normal data
# ==========================================================
final_scaler = MinMaxScaler().fit(np.concatenate([s["X"] for s in healthy_sessions]))
X_all_healthy = build_windows_from_sessions(healthy_sessions, final_scaler)

final_model = build_one_class_svm()
final_model.fit(flatten_windows(X_all_healthy))

FINAL_SCORE_THRESHOLD = calibrate_threshold(final_model, X_all_healthy)

print("\nFINAL SCORE THRESHOLD:", FINAL_SCORE_THRESHOLD)
print("All pure-normal windows:", X_all_healthy.shape)


In [ ]:
# ==========================================================
# CELL 13 — final model evaluation
# ==========================================================
rng = np.random.default_rng(SEED)

n_normal_eval = max(1, round(0.20 * len(healthy_sessions)))
normal_ids = rng.choice(len(healthy_sessions), size=n_normal_eval, replace=False)
selected_normal = [healthy_sessions[int(i)] for i in normal_ids]

fault_groups = {}
for session in fault_sessions:
    fault_groups.setdefault(session["fault_label"], []).append(session)

selected_faults = [group[int(rng.integers(len(group)))] for group in fault_groups.values()]
evaluation_sessions = selected_normal + selected_faults

print("\nFINAL EVALUATION FILES")
for session in evaluation_sessions:
    state = session.get("state", session.get("fault_label", "unknown"))
    print(f"label={session['binary_label']} | state={state} | file={session['file']}")

eval_file_df, eval_window_df = aggregate_predictions(
    evaluation_sessions, final_model, final_scaler, FINAL_SCORE_THRESHOLD
)

y_true = eval_window_df["true_label"].to_numpy()
y_pred = eval_window_df["pred_label"].to_numpy()
y_score = eval_window_df["anomaly_score"].to_numpy()

final_metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "normal_accuracy": np.mean(y_pred[y_true == 0] == 0),
    "anomaly_accuracy": np.mean(y_pred[y_true == 1] == 1),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_true, y_score),
    "score_threshold": FINAL_SCORE_THRESHOLD,
}

print("\nFINAL WINDOW-LEVEL RESULTS")
print(pd.Series(final_metrics))
print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"], zero_division=0))

eval_file_df.to_csv(os.path.join(FINAL_EVAL_DIR, "file_predictions.csv"), index=False)
eval_window_df.to_csv(os.path.join(FINAL_EVAL_DIR, "window_predictions.csv"), index=False)
pd.DataFrame([final_metrics]).to_csv(os.path.join(FINAL_EVAL_DIR, "metrics.csv"), index=False)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["normal window", "anomaly window"]).plot(values_format="d")
plt.title("Final One-Class SVM Window-Level Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(FINAL_EVAL_DIR, "confusion_matrix.png"), dpi=180)
plt.show()



In [ ]:
# ==========================================================
# CELL 14 — save model, scaler, threshold, and metadata
# ==========================================================
MODEL_PATH = os.path.join(FINAL_DIR, "pure_normal_one_class_svm.pkl")
SCALER_PATH = os.path.join(FINAL_DIR, "pure_normal_scaler.pkl")
THRESHOLD_PATH = os.path.join(FINAL_DIR, "one_class_svm_threshold.npz")
METADATA_PATH = os.path.join(FINAL_DIR, "model_metadata.csv")

joblib.dump(final_model, MODEL_PATH)
joblib.dump(final_scaler, SCALER_PATH)

np.savez(
    THRESHOLD_PATH,
    score_threshold=np.float32(FINAL_SCORE_THRESHOLD),
    file_anomaly_fraction_threshold=np.float32(FILE_ANOMALY_FRACTION_THRESHOLD),
)

pd.DataFrame({
    "feature_id": np.arange(N_FEATURES),
    "feature_name": FEATURE_COLUMNS,
    "scaler_min": final_scaler.data_min_,
    "scaler_max": final_scaler.data_max_,
}).to_csv(METADATA_PATH, index=False)

print("Saved:", MODEL_PATH)
print("Saved:", SCALER_PATH)
print("Saved:", THRESHOLD_PATH)
print("Saved:", METADATA_PATH)



In [ ]:
# ==========================================================
# CELL 15 — export One-Class SVM to ONNX
# ==========================================================
ONNX_PATH = os.path.join(FINAL_DIR, "pure_normal_one_class_svm.onnx")
input_size = WINDOW_LEN * N_FEATURES

onnx_model = convert_sklearn(
    final_model,
    initial_types=[("float_input", FloatTensorType([1, input_size]))],
    target_opset={"": 15, "ai.onnx.ml": 3},
)

onnx.checker.check_model(onnx_model)
onnx.save_model(onnx_model, ONNX_PATH)

print("Saved ONNX model:", ONNX_PATH)
print("ONNX size KB:", round(os.path.getsize(ONNX_PATH) / 1024, 2))


In [ ]:
# ==========================================================
# CELL 15.1 — export complete One-Class SVM as C header
# ==========================================================
HEADER_PATH = os.path.join(FINAL_DIR, "one_class_svm_model.h")

support_vectors = final_model.support_vectors_.astype(np.float32)
dual_coef = final_model.dual_coef_.ravel().astype(np.float32)
intercept = float(final_model.intercept_[0])
gamma = float(final_model._gamma)

with open(HEADER_PATH, "w", encoding="utf-8") as f:
    f.write("#ifndef ONE_CLASS_SVM_MODEL_H\n")
    f.write("#define ONE_CLASS_SVM_MODEL_H\n\n")
    f.write("#include <math.h>\n")
    f.write("#include <stdint.h>\n\n")

    f.write(f"#define OCSVM_N_FEATURES {N_FEATURES}\n")
    f.write(f"#define OCSVM_WINDOW_LEN {WINDOW_LEN}\n")
    f.write(f"#define OCSVM_INPUT_SIZE {WINDOW_LEN * N_FEATURES}\n")
    f.write(f"#define OCSVM_N_SUPPORT {len(support_vectors)}\n")
    f.write(f"#define OCSVM_SCALE_DIVISOR {SCALE_DIVISOR:.1f}f\n")
    f.write(f"#define OCSVM_GAMMA {gamma:.10f}f\n")
    f.write(f"#define OCSVM_INTERCEPT {intercept:.10f}f\n")
    f.write(f"#define OCSVM_SCORE_THRESHOLD {FINAL_SCORE_THRESHOLD:.10f}f\n")
    f.write(
        f"#define OCSVM_FILE_THRESHOLD "
        f"{FILE_ANOMALY_FRACTION_THRESHOLD:.10f}f\n\n"
    )

    f.write("static const float OCSVM_MIN[OCSVM_N_FEATURES] = {\n")
    for value in final_scaler.data_min_:
        f.write(f"    {float(value):.10f}f,\n")
    f.write("};\n\n")

    f.write("static const float OCSVM_MAX[OCSVM_N_FEATURES] = {\n")
    for value in final_scaler.data_max_:
        f.write(f"    {float(value):.10f}f,\n")
    f.write("};\n\n")

    f.write("static const float OCSVM_DUAL_COEF[OCSVM_N_SUPPORT] = {\n")
    for value in dual_coef:
        f.write(f"    {float(value):.10f}f,\n")
    f.write("};\n\n")

    f.write(
        "static const float "
        "OCSVM_SUPPORT_VECTORS[OCSVM_N_SUPPORT][OCSVM_INPUT_SIZE] = {\n"
    )

    for vector in support_vectors:
        values = ", ".join(f"{float(value):.10f}f" for value in vector)
        f.write(f"    {{{values}}},\n")

    f.write("};\n\n")

    f.write(
        "static inline float ocsvm_scale_raw("
        "float raw_value, int feature_id)\n"
    )
    f.write("{\n")
    f.write("    float physical = raw_value / OCSVM_SCALE_DIVISOR;\n")
    f.write("    float minv = OCSVM_MIN[feature_id];\n")
    f.write("    float maxv = OCSVM_MAX[feature_id];\n")
    f.write("    float denominator = maxv - minv;\n")
    f.write("    if (denominator == 0.0f) return 0.0f;\n")
    f.write("    return (physical - minv) / denominator;\n")
    f.write("}\n\n")

    f.write(
        "static inline float ocsvm_decision_function("
        "const float input[OCSVM_INPUT_SIZE])\n"
    )
    f.write("{\n")
    f.write("    float decision = OCSVM_INTERCEPT;\n")
    f.write("    for (int i = 0; i < OCSVM_N_SUPPORT; i++) {\n")
    f.write("        float distance_squared = 0.0f;\n")
    f.write("        for (int j = 0; j < OCSVM_INPUT_SIZE; j++) {\n")
    f.write(
        "            float difference = "
        "input[j] - OCSVM_SUPPORT_VECTORS[i][j];\n"
    )
    f.write("            distance_squared += difference * difference;\n")
    f.write("        }\n")
    f.write(
        "        decision += OCSVM_DUAL_COEF[i] * "
        "expf(-OCSVM_GAMMA * distance_squared);\n"
    )
    f.write("    }\n")
    f.write("    return decision;\n")
    f.write("}\n\n")

    f.write(
        "static inline float ocsvm_anomaly_score("
        "const float input[OCSVM_INPUT_SIZE])\n"
    )
    f.write("{\n")
    f.write("    return -ocsvm_decision_function(input);\n")
    f.write("}\n\n")

    f.write(
        "static inline int ocsvm_is_anomaly("
        "const float input[OCSVM_INPUT_SIZE])\n"
    )
    f.write("{\n")
    f.write(
        "    return ocsvm_anomaly_score(input) "
        "> OCSVM_SCORE_THRESHOLD;\n"
    )
    f.write("}\n\n")

    f.write("#endif\n")

print("Saved header:", HEADER_PATH)
print("Support vectors:", len(support_vectors))
print("Header size KB:", round(os.path.getsize(HEADER_PATH) / 1024, 2))

In [ ]:
# ==========================================================
# CELL 15.2 — create and download STM32 parameter header
# ==========================================================
import os

PARAM_HEADER_PATH = os.path.join(
    FINAL_DIR,
    "one_class_svm_params.h",
)

with open(PARAM_HEADER_PATH, "w", encoding="utf-8") as header:
    header.write("#ifndef ONE_CLASS_SVM_PARAMS_H\n")
    header.write("#define ONE_CLASS_SVM_PARAMS_H\n\n")

    header.write("/* One-Class SVM input configuration */\n")
    header.write(f"#define OCSVM_WINDOW_LEN {WINDOW_LEN}\n")
    header.write(f"#define OCSVM_N_FEATURES {N_FEATURES}\n")
    header.write(
        f"#define OCSVM_INPUT_SIZE "
        f"{WINDOW_LEN * N_FEATURES}\n\n"
    )

    header.write("/* Calibrated anomaly decision threshold */\n")
    header.write(
        f"#define OCSVM_SCORE_THRESHOLD "
        f"{float(FINAL_SCORE_THRESHOLD):.10f}f\n"
    )
    header.write(
        f"#define OCSVM_FILE_THRESHOLD "
        f"{float(FILE_ANOMALY_FRACTION_THRESHOLD):.10f}f\n\n"
    )

    header.write(
        "/* MinMaxScaler minimum values in physical units */\n"
    )
    header.write(
        "static const float "
        "OCSVM_MIN[OCSVM_N_FEATURES] = {\n"
    )

    for index, value in enumerate(final_scaler.data_min_):
        comma = "," if index < N_FEATURES - 1 else ""
        header.write(
            f"    {float(value):.10f}f{comma}  "
            f"/* {FEATURE_COLUMNS[index]} */\n"
        )

    header.write("};\n\n")

    header.write(
        "/* MinMaxScaler maximum values in physical units */\n"
    )
    header.write(
        "static const float "
        "OCSVM_MAX[OCSVM_N_FEATURES] = {\n"
    )

    for index, value in enumerate(final_scaler.data_max_):
        comma = "," if index < N_FEATURES - 1 else ""
        header.write(
            f"    {float(value):.10f}f{comma}  "
            f"/* {FEATURE_COLUMNS[index]} */\n"
        )

    header.write("};\n\n")
    header.write("#endif /* ONE_CLASS_SVM_PARAMS_H */\n")

print("Parameter header created:")
print(PARAM_HEADER_PATH)
print(
    "Header size:",
    round(os.path.getsize(PARAM_HEADER_PATH) / 1024, 2),
    "KB",
)

# Download automatically when running in Google Colab.
try:
    from google.colab import files as colab_files

    colab_files.download(PARAM_HEADER_PATH)

except ImportError:
    # Show a clickable link when running in Jupyter Notebook.
    try:
        from IPython.display import FileLink, display

        display(FileLink(PARAM_HEADER_PATH))

    except ImportError:
        print(
            "Automatic download is unavailable. "
            "Open the header from the path printed above."
        )

In [ ]:
# ==========================================================
# CELL 16 — manual test function
# ==========================================================
def manual_one_class_svm_prediction(test_file):
    model = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    threshold_data = np.load(THRESHOLD_PATH)

    threshold = float(threshold_data["score_threshold"])
    file_threshold = float(threshold_data["file_anomaly_fraction_threshold"])

    df = read_one(test_file)
    X_test, starts = make_windows_from_array(df.to_numpy(dtype=np.float32), scaler)
    scores = anomaly_score(model, X_test)
    flags = (scores > threshold).astype(np.int64)

    anomaly_fraction = float(np.mean(flags))
    file_decision = "ANOMALY" if anomaly_fraction >= file_threshold else "HEALTHY"

    result = pd.DataFrame({
        "window": np.arange(len(X_test)),
        "start_row": starts,
        "time_sec": (starts + WINDOW_LEN / 2.0) * ROW_PERIOD_SEC,
        "anomaly_score": scores,
        "score_threshold": threshold,
        "decision": np.where(flags == 1, "anomaly", "normal"),
        "is_anomaly": flags,
    })

    stem = Path(test_file).stem
    output_csv = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}.csv")
    result.to_csv(output_csv, index=False)

    print("\n" + "=" * 60)
    print("MANUAL ONE-CLASS SVM TEST")
    print("=" * 60)
    print("File:", test_file)
    print("Total windows:", len(result))
    print("Normal windows:", int((flags == 0).sum()))
    print("Anomalous windows:", int((flags == 1).sum()))
    print("Anomaly fraction:", round(anomaly_fraction, 6))
    print("File threshold:", file_threshold)
    print("FILE DECISION:", file_decision)
    print("Score threshold:", threshold)
    print("Saved CSV:", output_csv)
    print("\nWindow decisions:")
    print(result["decision"].value_counts())

    plt.figure(figsize=(12, 4))
    plt.plot(result["time_sec"], result["anomaly_score"])
    plt.axhline(threshold, linestyle="--", label="score threshold")
    plt.xlabel("Time [s]")
    plt.ylabel("One-Class SVM anomaly score")
    plt.title(f"One-Class SVM Score — {stem}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_score.png"), dpi=150)
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.step(result["time_sec"], result["is_anomaly"], where="mid")
    plt.yticks([0, 1], ["normal", "anomaly"])
    plt.ylim(-0.1, 1.1)
    plt.xlabel("Time [s]")
    plt.ylabel("Decision")
    plt.title(f"One-Class SVM Decision Timeline — {stem}")
    plt.tight_layout()
    plt.savefig(os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_timeline.png"), dpi=150)
    plt.show()

    return result


In [ ]:
# ==========================================================
# CELL 17 — select manual test file
# ==========================================================
if __name__ == "__main__":
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    TEST_FILE = filedialog.askopenfilename(
        title="Select motor CSV or Excel file",
        filetypes=[
            ("Excel files", "*.xlsx"),
            ("CSV files", "*.csv"),
            ("All files", "*.*"),
        ],
    )

    root.destroy()

    if TEST_FILE:
        manual_result = manual_one_class_svm_prediction(TEST_FILE)
        print(manual_result.head(30))
    else:
        print("No file selected.")